# El código esencial de un proyecto con datos

Técnicas Avanzadas de Predicción en Negocios · Tema 2

Mismo código que el deck interactivo, en el mismo orden. Ejecuta las celdas en orden con el entorno `tap`. Para cada celda, responde a las preguntas «Para pensar» modificando la misma celda.

**Solo si trabajas en Google Colab.** La celda siguiente instala las mismas versiones de los paquetes que el entorno `tap` del curso. En tu ordenador, con el entorno `tap` activado, no hace nada. Si Colab pide reiniciar la sesión después de instalar, acepta y vuelve a ejecutar el cuaderno desde el principio.

In [ ]:
# Instala las versiones del curso (solo en Google Colab)
import sys
if "google.colab" in sys.modules:
    %pip install -q -r https://raw.githubusercontent.com/juanDmontoro/tap/main/requirements.txt

# Preparar

## Carga de librerías (siempre al principio)

**Para pensar**

1. Añade al final la línea `Lasso()` y ejecuta. ¿Qué error aparece?
2. Corrígelo añadiendo `Lasso` a la línea que importa `Ridge`.

In [ ]:
# Tablas y cálculo numérico
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# División, validación cruzada y búsqueda de hiperparámetros
from sklearn.model_selection import train_test_split, KFold
from sklearn.model_selection import GridSearchCV, cross_val_score
# Transformación de columnas y pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
# Modelos y métricas
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import root_mean_squared_error, accuracy_score
from sklearn.metrics import log_loss, ConfusionMatrixDisplay
print("Librerías cargadas")

## Los datos: una tabla de anuncios

**Para pensar**

1. ¿Cuántas observaciones y cuántas columnas tiene la tabla?
2. Cambia `datos.head()` por `datos["room_type"].value_counts()`. ¿Cuántos anuncios son alojamientos enteros?
3. Prueba `datos.describe()`.

In [ ]:
# Dirección del archivo en la web del curso
URL = "https://juandmontoro.github.io/tap/data/airbnb_mini.csv"
# Leer el archivo como una tabla (DataFrame)
datos = pd.read_csv(URL)
# Seleccionar 1000 observaciones para este ejemplo
datos = datos.sample(n=1000, random_state=42)
# Número de filas y de columnas
print(datos.shape)
# Las cinco primeras filas
datos.head()

## ¿Qué predecimos y con qué?

**Para pensar**

1. ¿Cuántas observaciones hay en cada conjunto?
2. Cambia `test_size=0.3` por `0.2`. ¿Qué cambia?
3. Añade `"number_of_reviews"` a `numericas`. ¿Cuántas columnas tiene ahora `X`?

In [ ]:
# Respuesta: precio por noche
y = datos["price"]
# Predictores, separados por tipo
numericas = ["accommodates", "bedrooms", "bathrooms",
             "dist_km", "review_scores_rating"]
categoricas = ["room_type", "distrito"]
X = datos[numericas + categoricas]
# División: 70 % entrenamiento, 30 % test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)
print(X_train.shape, X_test.shape)

## Transformar las columnas según su tipo

**Para pensar**

1. `X_train` tiene 7 columnas. ¿Por qué salen 26?
2. Haz la asignación al principio del chunk a `categoricas=["room_type"]`. ¿Cuántas columnas salen? ¿Por qué?

In [ ]:
# Una transformación para cada tipo de columna
preprocesado = ColumnTransformer([
    # numéricas: media 0 y desviación típica 1
    ("numericas", StandardScaler(), numericas),
    # categóricas: una columna 0/1 por categoría
    ("categoricas", OneHotEncoder(handle_unknown="ignore"),
     categoricas),
])
# Aprender la transformación con el entrenamiento y aplicarla
X_transformada = preprocesado.fit_transform(X_train)
print(X_transformada.shape)

# Estimar y evaluar: regresión

## Un pipeline: transformación y modelo en un solo objeto

**Para pensar**

1. Compara las cinco predicciones con los cinco precios observados.
2. ¿Qué pasos tiene `pipe`? Añade una línea con `pipe` al final.

In [ ]:
# Pipeline: primero transforma, después estima
pipe = Pipeline([
    ("preprocesado", preprocesado),
    ("modelo", Ridge(alpha=1)),
])
# Estimar con el conjunto de entrenamiento
pipe.fit(X_train, y_train)
# Predecir el conjunto de test
pred_test = pipe.predict(X_test)
print(pred_test[:5].round(1))
print(y_test.values[:5])

## ¿Cuánto se equivoca el modelo?

**Para pensar**

1. ¿Mejora Ridge al modelo nulo en el test?
2. Cambia `Ridge(alpha=1)` por `Ridge(alpha=1000)` en la celda anterior, ejecútala y vuelve a ejecutar esta. ¿Qué pasa con los dos RMSE?

In [ ]:
# Predicciones en entrenamiento
pred_train = pipe.predict(X_train)
# RMSE (en euros) en cada conjunto
rmse_train = root_mean_squared_error(y_train, pred_train)
rmse_test = root_mean_squared_error(y_test, pred_test)
# Modelo nulo: predice siempre la media del entrenamiento
pred_nulo = np.full(len(y_test), y_train.mean())
rmse_nulo = root_mean_squared_error(y_test, pred_nulo)
print(f"Ridge, entrenamiento: {rmse_train:.1f} €")
print(f"Ridge, test:          {rmse_test:.1f} €")
print(f"Nulo, test:           {rmse_nulo:.1f} €")

## Elegir alpha por validación cruzada

**Para pensar**

1. ¿Por qué el nombre es `modelo__alpha`?
2. ¿Por qué hay un signo menos delante de `best_score_`?
3. Cambia `n_splits=5` por `10`. ¿Cambia el alpha elegido?

In [ ]:
# Valores de alpha que se prueban (11, de 0,01 a 1000)
rejilla = {"modelo__alpha": np.logspace(-2, 3, 11)}
# Cinco particiones del conjunto de entrenamiento
particiones = KFold(n_splits=5, shuffle=True, random_state=42)
# Búsqueda: un ajuste por alpha y por partición
busqueda = GridSearchCV(pipe, rejilla, cv=particiones,
                        scoring="neg_root_mean_squared_error")
busqueda.fit(X_train, y_train)
print("Mejor alpha:", busqueda.best_params_["modelo__alpha"].round(2))
print("RMSE de validación:", round(-busqueda.best_score_, 2))

## Mínimo del RMSE y regla del error típico

**Para pensar**

1. ¿Cuántos euros separan el mejor alpha del peor?
2. ¿Por qué la regla toma el alpha **más alto** dentro del límite?
3. Cambia `.max()` por `.min()`. ¿Qué alpha sale?

In [ ]:
# Resultados de la búsqueda como tabla
tabla = pd.DataFrame(busqueda.cv_results_)
tabla["alpha"] = tabla["param_modelo__alpha"]
tabla["rmse"] = -tabla["mean_test_score"]
tabla["error_tipico"] = tabla["std_test_score"] / np.sqrt(busqueda.n_splits_)
# Fila del mínimo y límite: mínimo + un error típico
minimo = tabla.loc[tabla["rmse"].idxmin()]
limite = minimo["rmse"] + minimo["error_tipico"]
# Alpha más alto (modelo más sencillo) dentro del límite
dentro = tabla[tabla["rmse"] <= limite]
print("Alpha del mínimo:", round(minimo["alpha"], 2))
print("Alpha por la regla:", round(dentro["alpha"].max(), 2))
print(tabla[["alpha", "rmse", "error_tipico"]].round(2))

## Evaluación anidada: medir el procedimiento completo

**Para pensar**

1. ¿Cuántos ajustes hace esta celda?
2. ¿Qué mide la CV anidada que el mínimo de la búsqueda no mide?

In [ ]:
# Cinco particiones externas, con otra semilla
externas = KFold(n_splits=5, shuffle=True, random_state=7)
# En cada una, la búsqueda completa elige alpha con las otras 4
puntuaciones = cross_val_score(busqueda, X_train, y_train,
                               cv=externas,
                               scoring="neg_root_mean_squared_error")
rmse_externo = -puntuaciones
print(rmse_externo.round(1))
print(f"CV anidada: {rmse_externo.mean():.2f} €")
print(f"Mínimo de la búsqueda: {-busqueda.best_score_:.2f} €")

# Estimar y evaluar: clasificación

## ¿Es superanfitrión?

**Para pensar**

1. ¿Qué líneas cambian respecto a la regresión?
2. ¿Qué hace `stratify=y`?

In [ ]:
# Respuesta 0/1 y predictores (ahora el precio es un predictor)
y = datos["superhost"]
numericas = ["accommodates", "bedrooms", "bathrooms", "dist_km",
             "review_scores_rating", "number_of_reviews", "price"]
X = datos[numericas + categoricas]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
# Mismo preprocesado, otro modelo
preprocesado = ColumnTransformer([
    ("numericas", StandardScaler(), numericas),
    ("categoricas", OneHotEncoder(handle_unknown="ignore"),
     categoricas)])
# Pipeline con un modelo logit
logit = Pipeline([("preprocesado", preprocesado),
                  ("modelo", LogisticRegression(max_iter=1000))])
logit.fit(X_train, y_train)
print("Proporción de superanfitriones en test:", round(y_test.mean(), 3))

## Clases y probabilidades

**Para pensar**

1. ¿Qué anuncio de los ocho se predice como superanfitrión? ¿Por qué?
2. Añade `clase_03 = (prob_test >= 0.3).astype(int)` y `print(clase_03[:8])`. ¿Cuántos 1 hay ahora?

In [ ]:
# Probabilidad estimada de ser superanfitrión
prob_test = logit.predict_proba(X_test)[:, 1]
# Clase predicha: 1 si la probabilidad es al menos 0,5
clase_test = logit.predict(X_test)
print(prob_test[:8].round(2))
print(clase_test[:8])

## ¿Mejora el logit al clasificador nulo?

**Para pensar**

1. ¿Qué exactitud consigue el nulo sin mirar ningún predictor?
2. ¿En qué métrica es mayor la ventaja del logit?

In [ ]:
# Clasificador nulo: probabilidad = proporción de 1 en entrenamiento
nulo = DummyClassifier(strategy="prior")
nulo.fit(X_train, y_train)
prob_nulo = nulo.predict_proba(X_test)[:, 1]
# Métricas en el conjunto de test
print("Exactitud, nulo: ", round(accuracy_score(y_test, nulo.predict(X_test)), 3))
print("Exactitud, logit:", round(accuracy_score(y_test, clase_test), 3))
print("Log-loss, nulo:  ", round(log_loss(y_test, prob_nulo), 3))
print("Log-loss, logit: ", round(log_loss(y_test, prob_test), 3))

## La matriz de confusión depende del umbral

**Para pensar**

1. Localiza VN, FP, FN y VP en la matriz.
2. Cambia `umbral = 0.5` por `0.3`. ¿Qué pasa con los VP y con los FP?

In [ ]:
# Umbral de decisión
umbral = 0.5
clase = (prob_test >= umbral).astype(int)
# Matriz de confusión en el conjunto de test
ConfusionMatrixDisplay.from_predictions(
    y_test, clase, display_labels=["neg", "pos"])
plt.show()

## Opcional: curva ROC y AUC

**Para pensar**

1. ¿Qué AUC tendría el clasificador nulo?
2. Cambia `prob_test` por `prob_nulo` en las dos líneas y compruébalo.

In [ ]:
from sklearn.metrics import RocCurveDisplay, roc_auc_score
# Curva ROC: todos los umbrales a la vez
RocCurveDisplay.from_predictions(y_test, prob_test)
plt.show()
# Área bajo la curva, en el conjunto de test
print("AUC:", round(roc_auc_score(y_test, prob_test), 3))